# lawgraph schema check
Point `DB` at your file and run all cells.

In [ ]:
import duckdb, pandas as pd
DB = "graph.duckdb"          # <-- your file
con = duckdb.connect(DB, read_only=True)

## 1. Expected tables exist, with row counts

In [ ]:
EXPECTED = {
 "node_court","node_docket","node_opinion","node_person","node_attorney","node_party",
 "edge_person_presides_over_docket","edge_person_author_opinion","edge_person_joined_by_opinion",
 "edge_attorney_attorney_in_docket","edge_party_party_in_docket","edge_attorney_represents_party",
 "edge_opinion_cites_opinion","edge_opinion_in_docket_docket","edge_docket_filed_in_court",
 "edge_person_sits_on_court","edge_person_succeeds_person"}
have = {r[0] for r in con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='main'").fetchall()}
print("missing:", sorted(EXPECTED - have) or "none")
print("extra:  ", sorted(have - EXPECTED) or "none")
pd.DataFrame([(t, con.execute(f"SELECT count(*) FROM {t}").fetchall()[0][0]) for t in sorted(EXPECTED & have)],
             columns=["table","rows"]).set_index("table")

## 2. Structural columns on every table

In [ ]:
NODE = {"id","uid","valid_from","valid_to","active_from","active_to"}
EDGE = {"src_id","dst_id","valid_from","valid_to","active_from","active_to"}
for t in sorted(EXPECTED & have):
    cols = {r[0] for r in con.execute(f"DESCRIBE {t}").fetchall()}
    need = NODE if t.startswith("node_") else EDGE
    print(f"{'OK ' if need <= cols else 'BAD'} {t:<36} missing={sorted(need - cols)}")

## 3. Attributes inherited from CourtListener

In [ ]:
for t in sorted(EXPECTED & have):
    attrs = [f"{r[0]}:{r[1]}" for r in con.execute(f"DESCRIBE {t}").fetchall()
             if r[0] not in NODE | EDGE]
    print(f"{t:<36} {', '.join(attrs)}")

## 4. Time coverage and undated rows

In [ ]:
rows = []
for t in sorted(EXPECTED & have):
    lo, hi, n, nodate, active_closed = con.execute(f"""
        SELECT min(valid_from), max(valid_from), count(*),
               count(*) FILTER (WHERE valid_from IS NULL),
               count(*) FILTER (WHERE active_to IS NOT NULL) FROM {t}""").fetchall()[0]
    rows.append((t, lo, hi, n, nodate, round(100*nodate/max(n,1),1), active_closed))
pd.DataFrame(rows, columns=["table","first","last","rows","no_date","no_date_%","active_to_set"]).set_index("table")

## 5. Sanity checks against known facts

In [ ]:
# SCOTUS bench on a few dates should be 9 (post-1869); earlier Courts were 5-7.
for d in ["1850-01-01","1900-01-01","1950-01-01","2000-01-01","2020-01-01"]:
    n = con.execute(f"""SELECT count(DISTINCT src_id) FROM edge_person_sits_on_court
                        WHERE dst_id='scotus' AND active_from <= DATE '{d}'
                          AND (active_to IS NULL OR active_to > DATE '{d}')""").fetchall()[0][0]
    print(f"SCOTUS sitting justices on {d}: {n}")

# Citations pointing forward in time (cited opinion filed after the citing one) should be rare.
bad, total = con.execute("""
    SELECT count(*) FILTER (WHERE o.date_filed > e.valid_from), count(*)
    FROM edge_opinion_cites_opinion e JOIN node_opinion o ON o.id = e.dst_id""").fetchall()[0]
print(f"anachronistic citations: {bad:,} / {total:,} ({100*bad/max(total,1):.2f}%)")

# Dangling endpoints: edges whose src/dst id is not in the node table.
for t, s, d in [("edge_opinion_cites_opinion","node_opinion","node_opinion"),
                ("edge_person_author_opinion","node_person","node_opinion"),
                ("edge_docket_filed_in_court","node_docket","node_court")]:
    ds, dd = con.execute(f"""SELECT count(*) FILTER (WHERE s.id IS NULL), count(*) FILTER (WHERE d.id IS NULL)
        FROM {t} e LEFT JOIN {s} s ON s.id=e.src_id LEFT JOIN {d} d ON d.id=e.dst_id""").fetchall()[0]
    print(f"{t:<34} dangling src={ds:,} dst={dd:,}")